In [1]:
# ── PDF & HTML → Markdown (external images) ──────────────────────────────────
# Drop PDFs / HTML files in the same folder as this notebook and run.
# Output:
#   files_as_md/<filename>.md        — one markdown file per source
#   files_as_md/images/<filename>_<img>.png — extracted images
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print('📦 Installing libraries...')
for pkg in ('pymupdf4llm', 'markdownify', 'beautifulsoup4', 'Pillow'):
    pip(pkg)
print('✅ Done.\n')

# ── imports ───────────────────────────────────────────────────────────────────
import re, shutil, base64
from pathlib import Path
import pymupdf4llm
from markdownify import markdownify
from bs4 import BeautifulSoup

OUTPUT_DIR = Path.cwd()
IMAGES_DIR = OUTPUT_DIR / 'pdf_images'
IMAGES_DIR.mkdir(exist_ok=True)

# ── PDF converter ─────────────────────────────────────────────────────────────
def convert_pdf(src: Path) -> Path:
    img_tmp_dir = IMAGES_DIR / f'_tmp_{src.stem}'
    img_tmp_dir.mkdir(exist_ok=True)

    md_text = pymupdf4llm.to_markdown(
        str(src),
        write_images=True,
        image_path=str(img_tmp_dir),
        image_format='png',
    )

    img_counter = [0]

    def save_and_link(match):
        raw = match.group(1)
        # Skip already-inline base64
        if raw.startswith('data:'):
            return match.group(0)
        img_src = Path(raw) if Path(raw).is_absolute() else img_tmp_dir / Path(raw).name
        if not img_src.exists():
            return match.group(0)
        img_counter[0] += 1
        new_name = f'{src.stem}_img_{img_counter[0]:03d}.png'
        dest = IMAGES_DIR / new_name
        shutil.copy2(img_src, dest)
        rel_path = dest.relative_to(OUTPUT_DIR)
        return f'![{new_name}]({rel_path})'

    md_text = re.sub(r'!\[.*?\]\((.+?)\)', save_and_link, md_text)

    # Clean up temp folder
    shutil.rmtree(img_tmp_dir, ignore_errors=True)

    out = OUTPUT_DIR / f'{src.stem}.md'
    out.write_text(md_text, encoding='utf-8')
    return out

# ── HTML converter ────────────────────────────────────────────────────────────
def convert_html(src: Path) -> Path:
    html = src.read_text(encoding='utf-8', errors='replace')
    soup = BeautifulSoup(html, 'html.parser')

    for tag in soup(['script', 'style', 'noscript', 'iframe']):
        tag.decompose()

    img_counter = 0
    for img_tag in soup.find_all('img'):
        img_src = img_tag.get('src', '')

        # Handle base64-embedded images
        if img_src.startswith('data:image/'):
            match = re.match(r'data:image/(\w+);base64,(.+)', img_src, re.DOTALL)
            if match:
                img_counter += 1
                ext = match.group(1).lower()
                ext = 'jpg' if ext == 'jpeg' else ext
                new_name = f'{src.stem}_img_{img_counter:03d}.{ext}'
                dest = IMAGES_DIR / new_name
                dest.write_bytes(base64.b64decode(match.group(2)))
                img_tag['src'] = f'images/{new_name}'
                img_tag['alt'] = img_tag.get('alt', new_name)
            continue

        # Handle local file images
        img_path = (src.parent / img_src).resolve()
        if img_path.exists():
            img_counter += 1
            ext = img_path.suffix.lstrip('.').lower() or 'png'
            new_name = f'{src.stem}_img_{img_counter:03d}.{ext}'
            dest = IMAGES_DIR / new_name
            shutil.copy2(img_path, dest)
            img_tag['src'] = f'images/{new_name}'
            img_tag['alt'] = img_tag.get('alt', new_name)

    md_text = markdownify(str(soup), heading_style='ATX', bullets='-')
    md_text = re.sub(r'\n{3,}', '\n\n', md_text).strip()

    out = OUTPUT_DIR / f'{src.stem}.md'
    out.write_text(md_text, encoding='utf-8')
    return out

# ── scan & convert ────────────────────────────────────────────────────────────
HERE  = Path('.').resolve()
files = [
    f for f in HERE.iterdir()
    if f.is_file()
    and f.suffix.lower() in {'.pdf', '.html', '.htm'}
]

if not files:
    print('⚠️  No PDF or HTML files found in:', HERE)
else:
    print(f'🔍 Found {len(files)} file(s):\n')
    for src in files:
        print(f'  {src.name}  →  ', end='', flush=True)
        try:
            out     = convert_pdf(src) if src.suffix.lower() == '.pdf' else convert_html(src)
            size_kb = round(out.stat().st_size / 1024)
            print(f'✅  {out.name}  ({size_kb} KB)')
        except Exception as e:
            print(f'❌  {e}')
    print(f'\n🎉 All done → {OUTPUT_DIR}/')

📦 Installing libraries...
✅ Done.

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
🔍 Found 2 file(s):

  prof_guide_professor_notes.pdf  →  ✅  prof_guide_professor_notes.md  (4 KB)
  proj_guide_NLPS_-_TREC_2025_BioGen.pdf  →  ✅  proj_guide_NLPS_-_TREC_2025_BioGen.md  (12 KB)

🎉 All done → c:\Users\franc\Desktop\NLP_Biomedical_Agent\_proj_guides/
